# Test the Chinook SQL Agent Connection

This notebook checks the pieces used by `sql_agent.py`: environment loading, OpenAI model access, MySQL connectivity, direct SQL queries, and a final natural-language SQL agent call.

## 1. Load Environment Variables

The notebook checks for `.env` in the current working directory first, then in the repository root. The MySQL values can be omitted if your local defaults are `root`, no password, `localhost`, port `3306`, and database `Chinook`.

In [7]:
from pathlib import Path
import os

from dotenv import load_dotenv

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "requirements.in").exists():
            return path
    raise RuntimeError("Could not find the repository root from the current notebook directory.")


current_dir = Path.cwd().resolve()
repo_root = find_repo_root(current_dir)

# Load current-directory values first; the repo-root .env fills only missing values.
for env_path in (current_dir / ".env", repo_root / ".env"):
    if env_path.exists():
        load_dotenv(env_path, override=False)

print("Current directory:", current_dir)
print("Repository root:", repo_root)

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("MYSQL_HOST:", os.getenv("MYSQL_HOST", "localhost"))
print("MYSQL_DATABASE:", os.getenv("MYSQL_DATABASE", "Chinook"))

Current directory: /Users/mxagar/nexo/git_repositories/agents_guide/01_Fundamentals/lab/07_sql_agent_project
Repository root: /Users/mxagar/nexo/git_repositories/agents_guide
OPENAI_API_KEY loaded: True
MYSQL_HOST: localhost
MYSQL_DATABASE: Chinook


## 2. Test the OpenAI Chat Model

In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    temperature=float(os.getenv("OPENAI_TEMPERATURE", "0")),
    max_tokens=int(os.getenv("OPENAI_MAX_TOKENS", "512")),
)

llm.invoke("Reply with only: model connection ok").content

'model connection ok'

## 3. Connect to MySQL with LangChain

In [9]:
from urllib.parse import quote_plus

from langchain_community.utilities.sql_database import SQLDatabase

mysql_username = os.getenv("MYSQL_USERNAME", "root")
mysql_password = os.getenv("MYSQL_PASSWORD", "")
mysql_host = os.getenv("MYSQL_HOST", "localhost")
mysql_port = os.getenv("MYSQL_PORT", "3306")
database_name = os.getenv("MYSQL_DATABASE", "Chinook")

mysql_uri = (
    f"mysql+mysqlconnector://{quote_plus(mysql_username)}:{quote_plus(mysql_password)}"
    f"@{mysql_host}:{mysql_port}/{database_name}"
)

db = SQLDatabase.from_uri(mysql_uri)
db.dialect, db.get_usable_table_names()[:5]

('mysql', ['Album', 'Artist', 'Customer', 'Employee', 'Genre'])

## 4. Run Direct SQL Checks

In [10]:
print(db.run("SELECT COUNT(*) AS album_count FROM Album;"))
print(db.run("SELECT Name FROM Artist ORDER BY ArtistId LIMIT 5;"))

[(347,)]
[('AC/DC',), ('Accept',), ('Aerosmith',), ('Alanis Morissette',), ('Alice In Chains',)]


## 5. Create and Test the SQL Agent

In [11]:
from langchain_classic.agents import AgentType
from langchain_community.agent_toolkits import create_sql_agent

agent_executor = create_sql_agent(
    llm=llm,
    db=db,
    verbose=True,
    handle_parsing_errors=True,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
)

result = agent_executor.invoke({"input": "How many albums are in the database?"})
result["output"]



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: 
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, TrackThought: The table "Album" is likely the one that contains the albums. I should check the schema of the Album table to confirm the relevant column for counting albums.
Action: sql_db_schema
Action Input: Album
CREATE TABLE `Album` (
	`AlbumId` INTEGER NOT NULL, 
	`Title` VARCHAR(160) CHARACTER SET utf8mb3 COLLATE utf8mb3_general_ci NOT NULL, 
	`ArtistId` INTEGER NOT NULL, 
	PRIMARY KEY (`AlbumId`), 
	CONSTRAINT `FK_AlbumArtistId` FOREIGN KEY(`ArtistId`) REFERENCES `Artist` (`ArtistId`)
)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/Thought: The Album table contains AlbumId, Title, and ArtistId. To find the number of albums, I just need to co

'There are 347 albums in the database.'